# Chapter 15 Exercise 1 - Partial Siberian Snakes

This notebook implements the first exercise from Chapter 15 using SciBmad. The goal is to add the two partial Siberian snakes at the existing `CSNK` and `WSNK` marker locations and then check that the closed-orbit spin tune is no longer simply $G\gamma$.

## Load the AGS-like Spin Lattice

The tutorial lattice already contains the two marker elements `csnk` and `wsnk`. In this exercise those markers are converted into thin spin-rotation map elements.

In [ ]:
using SciBmad
using CairoMakie
using Printf

include(joinpath(pkgdir(SciBmad), "lattices", "ags.jl"))

snake_marker_indices = findall(ele -> ele.name in ("csnk", "wsnk"), ring.line)
for idx in snake_marker_indices
    @printf("%4d  %-6s  kind = %s\n", idx, ring.line[idx].name, ring.line[idx].kind)
end

## Taylor Element Equivalent in SciBmad

In Bmad, a Taylor element directly specifies the map of an element. For this exercise the orbital part of the map is the identity, while the spin part is an instantaneous quaternion rotation.

In SciBmad, the same idea is implemented with a `LineElement` map, using the `transport_map` property. The map function receives the orbital coordinates `v` and the accumulated spin quaternion `q`, then returns the updated pair `(v_out, q_out)`.

## Snake Quaternion

A unit quaternion

$$
q = \left(\cos\frac{\theta}{2}, \sin\frac{\theta}{2}\vec e\right)
$$

rotates a spin vector by angle $\theta$ around the unit vector $\vec e$. Both snakes rotate around the longitudinal axis, so $\vec e = \hat z$ and

$$
q_\mathrm{snake} = \left(\cos\frac{\theta}{2}, 0, 0, \sin\frac{\theta}{2}\right).
$$

The cold snake has $\theta = 18.3^\circ$, and the warm snake has $\theta = 10.57^\circ$.

In [ ]:
function quat_mul_tuple(q1, q2)
    a1, b1, c1, d1 = q1
    a2, b2, c2, d2 = q2
    return (
        a1*a2 - b1*b2 - c1*c2 - d1*d2,
        a1*b2 + b1*a2 + c1*d2 - d1*c2,
        a1*c2 - b1*d2 + c1*a2 + d1*b2,
        a1*d2 + b1*c2 - c1*b2 + d1*a2,
    )
end

function longitudinal_snake_quaternion(angle_deg)
    theta = deg2rad(angle_deg)
    return (cos(theta/2), 0.0, 0.0, sin(theta/2))
end

q_cold = longitudinal_snake_quaternion(18.3)
q_warm = longitudinal_snake_quaternion(10.57)

@printf("cold snake q = (%.8f, %.8f, %.8f, %.8f)\n", q_cold...)
@printf("warm snake q = (%.8f, %.8f, %.8f, %.8f)\n", q_warm...)

## Build the Thin Snake Map

The snake map leaves the orbital coordinates unchanged. The spin quaternion is updated by left-multiplying the snake quaternion onto the accumulated spin quaternion. This matches the ordering used internally by SciBmad spin tracking.

In [ ]:
function longitudinal_snake_map(v, q, qsnake)
    if isnothing(q)
        return v, nothing
    end
    return v, quat_mul_tuple(qsnake, q)
end

function add_longitudinal_snake!(bl, marker_name, angle_deg)
    idx = findfirst(ele -> ele.name == marker_name, bl.line)
    isnothing(idx) && error("Marker $(marker_name) not found")

    qsnake = longitudinal_snake_quaternion(angle_deg)
    bl.line[idx].transport_map = longitudinal_snake_map
    bl.line[idx].transport_map_params = qsnake
    bl.line[idx].kind = "Map"
    return idx, qsnake
end

## Insert the Cold and Warm Snakes

Work on a copy of the original ring. The original `ring` is kept unchanged so we can compare the spin tune with and without snakes.

In [ ]:
ring_snakes = deepcopy(ring)

cold_snake = add_longitudinal_snake!(ring_snakes, "csnk", 18.3)
warm_snake = add_longitudinal_snake!(ring_snakes, "wsnk", 10.57)

@printf("Cold snake inserted at element %d\n", cold_snake[1])
@printf("Warm snake inserted at element %d\n", warm_snake[1])

## Estimate the One-Turn Spin Tune

To mimic the `show spin` check, track a reference particle with an identity spin quaternion for one turn. The final quaternion is the closed-orbit one-turn spin map. If

$$
q_0 = \cos(\pi Q_s),
$$

then $Q_s$ is the spin tune modulo the usual integer/sign ambiguity. The helper below reports the representative tune in the range $0 \le Q_s \le 0.5$.

In [ ]:
const SNAKE_SPECIES = Species("proton")
const SNAKE_G = gyromagnetic_anomaly(SNAKE_SPECIES)
const SNAKE_MASS = massof(SNAKE_SPECIES)

energy_from_Ggamma_snake(Ggamma) = Ggamma / SNAKE_G * SNAKE_MASS

function p_over_q_reference(bl)
    try
        return bl.p_over_q_ref(0)
    catch
        return bl.p_over_q_ref
    end
end

function one_turn_spin_tune!(bl, Ggamma)
    bl.E_ref = energy_from_Ggamma_snake(Ggamma)
    b = Bunch(zeros(1, 6), [1.0 0.0 0.0 0.0];
        species=bl.species_ref,
        p_over_q_ref=p_over_q_reference(bl),
    )
    track!(b, bl)
    q = vec(b.coords.q)
    spin_tune = acos(clamp(abs(q[1]), -1, 1)) / pi
    return spin_tune, q
end

function nearest_integer_spin_tune(Ggamma)
    frac = mod(Ggamma, 1)
    return min(frac, 1 - frac)
end

## Compare with the No-Snake Value

In a perfectly flat ring without snakes, the closed-orbit spin tune is $G\gamma$ modulo an integer. With the two partial snakes installed, the one-turn spin map includes both the usual arc precession and the two longitudinal spin rotations. The resulting tune is therefore not simply $G\gamma$.

In [ ]:
Ggamma_samples = [51.71, 51.81, 51.91]

@printf("%8s  %14s  %18s  %14s\n", "Ggamma", "snake Qs", "no-snake |frac|", "difference")
for Ggamma in Ggamma_samples
    Qs_snake, q = one_turn_spin_tune!(ring_snakes, Ggamma)
    Qs_flat = nearest_integer_spin_tune(Ggamma)
    @printf("%8.2f  %14.8f  %18.8f  %14.8e\n",
        Ggamma, Qs_snake, Qs_flat, Qs_snake - Qs_flat)
end

## Scan the Energy

The scan below makes the difference easier to see. The snake tune follows the full one-turn spin map, not the flat-ring relation $Q_s = G\gamma$.

In [ ]:
Ggamma_grid = range(51.0, 52.0, length=101)
Qs_snake = similar(collect(Ggamma_grid))
Qs_flat = nearest_integer_spin_tune.(Ggamma_grid)

for (i, Ggamma) in enumerate(Ggamma_grid)
    Qs_snake[i], _ = one_turn_spin_tune!(ring_snakes, Ggamma)
end

fig = Figure(size=(760, 440))
ax = Axis(fig[1, 1], xlabel="Ggamma", ylabel="spin tune representative")
lines!(ax, Ggamma_grid, Qs_flat, label="flat ring: Ggamma modulo integer", linewidth=2)
lines!(ax, Ggamma_grid, Qs_snake, label="with partial snakes", linewidth=2)
axislegend(ax, position=:lt)
fig

## Why the Spin Tune Changes

Without snakes, the spin sees only vertical magnetic fields on the closed orbit, so the one-turn spin rotation is a simple precession with tune $G\gamma$. Once the partial snakes are added, the one-turn spin map is the product of several rotations: arc precession plus a cold-snake longitudinal rotation plus more arc precession plus a warm-snake longitudinal rotation. Since rotations about different axes generally do not commute, the rotation angle of the full one-turn map is not equal to the simple flat-ring value $G\gamma$.